In [ ]:
from database.db import SessionLocal
from database.models import Employee, Investigation

session = SessionLocal()
employees = session.query(Employee).all()
for emp in employees:
    print(emp.id, emp.name, emp.policy_limit, emp.risk_tier, emp.manager_slack_id)
print(Investigation)
session.close()

In [ ]:
from database.models import ExpensePolicy
policy = session.query(ExpensePolicy).all()
for p in policy:
    print(p.category, p.max_amount, p.requires_receipt, p.note)
session.close()


In [ ]:
from database.db import engine
from sqlalchemy import text

def run_raw_sql(query: str):
    """Executes a raw SQL query and prints results."""
    with engine.connect() as conn:
        result = conn.execute(text(query))
        for row in result:
            print(row)
    conn.close()

# Example usage:

run_raw_sql("SELECT * FROM transactions-- where risk_tier = 'low'")

In [ ]:
import json

def get_employee_expense_profile(employee_id: str):
    """Return employee profile as a JSON string for the LLM's observation."""
    session = SessionLocal()
    try:
        employee = session.query(Employee).filter(Employee.id == employee_id).first()
        # employee profile as a JSON string
        employee_profile = {"employee_id": employee.id,
            "name": employee.name,
            "policy_limit": employee.policy_limit,
            "risk_tier": employee.risk_tier,
            "manager_slack_id": employee.manager_slack_id}
        return json.dumps(employee_profile)

    except Exception as e:
        """Return an error string the LLM can understand"""
        return json.dumps({"error": f"Database error: {str(e)}"})

    finally:
        session.close()

get_employee_expense_profile("E456")

In [ ]:
import datetime
from zoneinfo import ZoneInfo

# Get the current time explicitly in IST
today = datetime.datetime.today().date()

print(today)

In [ ]:

import os
import requests
from dotenv import load_dotenv

load_dotenv()

# for Telegram
telegram_bot_token = os.getenv("TELEGRAM_TOKEN")
telegram_chatid = os.getenv("TELEGRAM_CHATID")

message = "Hii Dude! "

def send_telegram_escalation(message):
    url = f"https://api.telegram.org/bot{telegram_bot_token}/sendMessage"

    # escalation_message = create_escalation_message(employee_id, transaction_details, escalation_reason, evidence_summary, employee_slack_id)

    # Using MarkdownV2 or HTML is much more stable for multi-agent variables
    payload = {"chat_id": telegram_chatid, "text": message, "parse_mode": "Markdown"}

    try:
        # Always use json= payload for streaming LLM text to avoid form-encoding corruption
        response = requests.post(url, json=payload)
        response.raise_for_status() 
        return {"status": "Success", "platform": "Telegram"}
    except Exception as e:
        if 'response' in locals() and response is not None:
            return {"status": "Error", "message": f"Telegram Rejected Payload: {response.text}"}
        return {"status": "Error", "message": str(e)}

f = send_telegram_escalation(message)
print(f)

In [ ]:
from database.common import run_sql_query

data = run_sql_query("SELECT * FROM investigations-- where risk_tier = 'low'")

In [ ]:
# for d in data:
#     print(d)

for r in data:
    print(r)


In [ ]:
from database.db import SessionLocal
from database.models import Employee, Transaction

session = SessionLocal()

# Fetch one employee
emp = session.query(Employee).filter(Employee.id == "E456").first()

# 1. Default lazy="select" (or omit lazy parameter)
# Access transactions – watch the queries
print(emp.transactions)   # This triggers a SELECT now
print(emp.transactions)   # Second access uses cached list – no query

In [ ]:
q = emp.transactions.filter(Transaction.amount > 100)
print(q.all())

In [ ]:
from database.db import engine
from sqlalchemy import text

with engine.connect() as conn:
    plan = conn.execute(text("EXPLAIN QUERY PLAN SELECT * FROM transactions WHERE employee_id = 'E456' AND date >= '2026-01-01'")).fetchall()
    for row in plan:
        print(row)